In [ ]:
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
from rich import print

/rds/general/user/ztb25/home/miniforge3/envs/integrate/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/rds/general/user/ztb25/home/miniforge3/envs/integrate/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# load in dataset
adata = sc.read("/rds/general/user/ztb25/home/PBMC_datasets/integrated_adata.h5ad")

In [9]:
adata

AnnData object with n_obs × n_vars = 428373 × 14513
    obs: 'diagnosis', 'age', 'gender', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'celltypist_cell_label_coarse', 'celltypist_conf_score_coarse', 'celltypist_cell_label_fine', 'celltypist_conf_score_fine', 'dataset', 'sample_combined'
    var: 'feature_types', 'mt', 'gene_symbols', 'genome', 'gene_versions', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'bbknn', 'celltypist_cell_label_coarse_colors', 'celltypist_cell_label_fine_colors', 'dataset_colors', 'harmony', 'hvg', 'neighbors', 'neighbors_unintegrated', 'pca', 'sample_combined_colors', 'scANVI', 'scanorama', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scANVI', 'X_scVI', 'X_scanorama', 'X_umap', 'X_umap_bbknn', 'X_umap_harmony', 'X_umap_scANVI', 'X_umap_scVI', 'X_

In [ ]:
# rename obsm keys for plotting
adata.obsm["Harmony"] = adata.obsm["X_pca_harmony"]
adata.obsm["scVI"] = adata.obsm["X_scVI"]
adata.obsm["scANVI"] = adata.obsm["X_scANVI"]
adata.obsm["Scanorama"] = adata.obsm["X_scanorama"]
adata.obsm["Unintegrated"] = adata.obsm["X_pca"]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key="dataset",
    label_key="celltypist_cell_label_fine",
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=["Harmony","Unintegrated", "scVI", "scANVI", "Scanorama"],
    n_jobs=16,
)
bm.benchmark()

In [ ]:
bm.plot_results_table(save_dir="/rds/general/user/ztb25/home/Figures/Unscaled", min_max_scale=False)

In [ ]:
bm.plot_results_table(save_dir="/rds/general/user/ztb25/home/Figures/Scaled", min_max_scale=True)

In [ ]:
df = bm.get_results(min_max_scale=False)
print(df)

In [ ]:
df.transpose()

In [ ]:
df2 = bm.get_results(min_max_scale=True)
print(df2)

In [ ]:
df2.transpose()